In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
# LOCAL - So can access functions in isca_tools which is in StAndrews/Isca
sys.path.append('/home/users/jamd1/Isca/')
from isca_tools import cesm
from isca_tools.utils import area_weighting, print_ds_var_list
from isca_tools.utils.constants import rho_water
import cftime
import matplotlib.units
from matplotlib.dates import ConciseDateConverter
matplotlib.units.registry[cftime.DatetimeNoLeap] = ConciseDateConverter()  # to allow plotting time as x axis
plt.style.use('/home/users/jamd1/Isca/jobs/publish.mplstyle')

In [2]:
# Latent heat of vaporization (or condensation) at 0 Celsius. Units: Jkg^{-1}
L_v = 2.5e6

# Latent heat of sublimation. Units: Jkg^{-1}
L_sub = 2.834E6

# Latent heat of fusion. Units: Jkg^{-1}
L_fus = L_sub - L_v

In [3]:
ar = 4/3        # aspect ratio (width/height)
# Details required for Journal of Climate Figures
low_dpi = 100
dpi = {'monochrome': 1100, 'combination': 800, 'halftone': 300}
width = {'one_col': 3.2, 'two_col': 5.5}        # width in inches 
save_pad_inches = 0.05

# Default parameters
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
ax_linewidth = plt.rcParams['axes.linewidth']

In [4]:
def ds_month_shift(ds, decode_times=True):
    # ds should be loaded with decode_times=False
    # First month is marked as February not January, need to shift every month by -1
    n_day_month = np.asarray([cftime.DatetimeNoLeap(1, i+1, 1).daysinmonth for i in range(12)])         # number of days in each month
    n_months_in_ds = ds.time.size
    n_years_in_ds = int(np.floor(n_months_in_ds/12))
    month_shift_array = np.concatenate((np.tile(n_day_month, n_years_in_ds),
                                        n_day_month[:n_months_in_ds%12]))
    ds_new=ds.assign_coords({'time':('time',ds.time.values-month_shift_array,ds.time.attrs)})
    if decode_times:
        ds_new = xr.decode_cf(ds_new)
    return ds_new

In [6]:
# exp_name = 'e.e20.E1850TEST.f09_g17.test'
exp_name = 'e.e20.E1850TEST.f09_g17.co2_4x'
var_keep = ['TS', 'FSNT', 'FLNT', 'gw', 'LHFLX', 'SHFLX', 'FLNS', 'FSNS', 'PRECSC', 'PRECSL']
ds = cesm.load_dataset(exp_name, decode_times=False)[var_keep].load()
ds = cesm.load_dataset(exp_name, decode_times=False)[var_keep].load()
ds = ds_month_shift(ds)     # Correct for 1st month being Feb not Jan

In [11]:
# import sys
# sys.path.append('/home/users/jamd1/Isca/')
# import isca_tools
# # var_keep = ['TS', 'FSNT', 'FLNT', 'gw', 'LHFLX', 'SHFLX', 'FLNS', 'FSNS', 'PRECSC', 'PRECSL']
# exp_name = 'e.e20.ETEST.f19_g17.test'
# ds = isca_tools.cesm.load_dataset(exp_name, decode_times=False).load()
# isca_tools.utils.print_ds_var_list(ds, 'temp')
# ds.TS.plot()

## Spin up plot
If spin up finished, shoudl get net flux imbalance of 0 in bottom plot. We do not see this.

In [7]:
weight = ds.gw.isel(time=0)
# weight = np.cos(np.deg2rad(ds.lat))
weight = weight / weight.mean()

In [ ]:
time_array = ds.time.to_numpy()
temp_global_mean = (weight * ds.TS).mean(dim=['lat', 'lon'])
temp_annual_mean = temp_global_mean.groupby('time.year').mean()

flux_tom_global_mean = (weight * (ds.FSNT-ds.FLNT)).mean(dim=['lat', 'lon'])    # I think this is net upward value
flux_tom_annual_mean = flux_tom_global_mean.groupby('time.year').mean()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(12, 5), sharex=True)
# ax[0].plot(time_array, temp_global_mean, color='k')
ax[0].plot(time_array[5::12], temp_annual_mean, color=default_colors[0], marker='|', markersize=20)
# ax[1].plot(time_array, flux_tom_global_mean, color='k')
ax[1].plot(time_array[5::12], flux_tom_annual_mean, color=default_colors[0], marker='|', markersize=20)
ax[1].axhline(0, color='k', lw=ax_linewidth)
ax[0].set_ylabel('Surface Temperature [K]')
ax[1].set_ylabel('Net TOM SW - LW Flux [Wm$^{-2}$]')
ax[0].set_xlim(time_array[0], time_array[-1])
ax[0].set_xticks(time_array[5::12]);
fig.savefig(f"spin_up_tom.pdf", dpi=dpi['combination'], bbox_inches='tight', pad_inches=save_pad_inches)

In [ ]:
print(float(flux_tom_annual_mean[-1]))

## Surface Energy Budget
Comes from [online tutorial](https://brian-rose.github.io/ClimateLaboratoryBook/courseware/surface-energy-balance.html).

In [ ]:
def get_net_surface_energy_budget(ds):
    # Positive is upward flux
    net_rad = ds.FLNS - ds.FSNS      # add net longwave and net shortwave at surface
    turb_flux = ds.LHFLX + ds.SHFLX  # add latent and sensible heat fluxes
    snow_flux = (ds.PRECSC + ds.PRECSL) * rho_water * L_fus
    return net_rad + turb_flux + snow_flux
    
ds['net_sfc_flux'] = get_net_surface_energy_budget(ds)
flux_surf_global_mean = (weight * ds['net_sfc_flux']).mean(dim=['lat', 'lon'])
flux_surf_annual_mean = flux_surf_global_mean.groupby('time.year').mean()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 5), sharex=True)
# ax.plot(time_array, flux_tom_global_mean, color=default_colors[0], lw=ax_linewidth)
# ax.plot(time_array, -flux_surf_global_mean, color=default_colors[3], lw=ax_linewidth)
ax.plot(time_array[5::12], flux_tom_annual_mean, color=default_colors[0], marker='.', markersize=5, label='TOM')
ax.plot(time_array[5::12], -flux_surf_annual_mean, color=default_colors[3], marker='.', markersize=5, label='Surface')
ax.axhline(0, color='k', lw=ax_linewidth)
ax.set_ylabel('Net Flux [Wm$^{-2}$]')
ax.set_xlim(time_array[0], time_array[-1])
ax.set_xticks(time_array[5::12])
ax.legend()
# fig.savefig(f"spin_up_sfc.pdf", dpi=dpi['combination'], bbox_inches='tight', pad_inches=save_pad_inches)

## Q Flux
Below, we see that the net upward surface energy flux looks a lot like the Q flux in the input file. [🔗](https://brian-rose.github.io/ClimateLaboratoryBook/courseware/introducing-cesm.html#ocean-boundary-conditions)

They are hard to combine as on different grids.

In [ ]:
input_dir = '/gws/nopw/j04/global_ex/jamd1/cesm/CESM2.1.3/cesm_inputdata/ocn/docn7/SOM/'
exp_input_file = 'pop_frc.b.e21.B1850.f09_g17.CMIP6-piControl.001_branch2.012120.nc'
ds_input = xr.load_dataset(input_dir+exp_input_file)[['qdp', 'area']].load()

In [ ]:
(-ds_input.qdp.mean(dim='time')).plot()
plt.title('Prescribed heat flux out of ocean (Wm$^{-2}$), annual mean')

In [ ]:
# Annual mean surface energy budget looks like q flux
ds['net_sfc_flux'].groupby('time.year').mean()[-1].plot()
plt.title(f'Net upward Surface Flux in Year {ds.time.to_numpy()[-1].year}');